# AM01 Official Experiments

Notebook ufficiale Colab per il progetto AM01: dataset inspection, preprocessing, main protocol, ablation essenziali, diagnostica AAE e figure finali per il report.

Protocollo principale: `window_length=64`, `stride=16`, StandardScaler, MSE, seed 42. La sensitivity usa solo `w=32` e `w=64`; `w=128` e' escluso perche' riduce troppo il numero di finestre valutabili.

## 0. Parametri

In [1]:
from pathlib import Path

REPO_URL = "https://github.com/Bernuz2003/AML_anomaliy_detection.git"  # opzionale: https://github.com/<user>/<repo>.git
PROJECT_DIR = Path('/content/am01-kuka-aae-anomaly-detection')
DRIVE_ROOT = Path('/content/drive/MyDrive/AM01')
DATA_DIR = DRIVE_ROOT / 'data' / 'KukaVelocityDataset'
OFFICIAL_ROOT = DRIVE_ROOT / 'results' / 'official'
CONFIG_DIR = OFFICIAL_ROOT / 'config'
TABLES_DIR = OFFICIAL_ROOT / 'tables'
FIGURES_DIR = OFFICIAL_ROOT / 'figures'
RUNS_DIR = OFFICIAL_ROOT / 'runs'
EXTENDED_DIR = OFFICIAL_ROOT / 'extended_scores'

RUN_PREPROCESSING = True
RUN_MAIN_EXPERIMENTS = True
RUN_CORE_ABLATIONS = True
RUN_AAE_DIAGNOSTICS = True
RUN_REPORT_FIGURES = True

PRIMARY_WINDOW_LENGTH = 64
PRIMARY_STRIDE = 16
SENSITIVITY_WINDOWS = [32, 64]
SEEDS_MAIN = [42]
SEEDS_STABILITY = [0, 1, 2]
SELECTION_METRIC = 'val_pr_auc'

print('DATA_DIR:', DATA_DIR)
print('OFFICIAL_ROOT:', OFFICIAL_ROOT)

DATA_DIR: /content/drive/MyDrive/AM01/data/KukaVelocityDataset
OFFICIAL_ROOT: /content/drive/MyDrive/AM01/results/official


## 1. Mount Drive e setup repo

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import subprocess
import sys


def sh(cmd: str) -> None:
    print(f"\n$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

for path in [OFFICIAL_ROOT, CONFIG_DIR, TABLES_DIR, FIGURES_DIR, RUNS_DIR, EXTENDED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not PROJECT_DIR.exists():
    if not REPO_URL:
        raise RuntimeError('PROJECT_DIR non esiste. Imposta REPO_URL o carica il repo in /content.')
    sh(f'git clone {REPO_URL} "{PROJECT_DIR}"')

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
print('Working directory:', Path.cwd())

sh('pip install -q -r requirements.txt')
sh('python -m compileall -q src scripts')
sh(f'pip freeze > "{CONFIG_DIR / "environment.txt"}"')


$ git clone https://github.com/Bernuz2003/AML_anomaliy_detection.git "/content/am01-kuka-aae-anomaly-detection"
Working directory: /content/am01-kuka-aae-anomaly-detection

$ pip install -q -r requirements.txt

$ python -m compileall -q src scripts

$ pip freeze > "/content/drive/MyDrive/AM01/results/official/config/environment.txt"


## 2. Import e configurazione ufficiale

In [4]:
import json
import shutil

import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from am01.utils.config import load_config
import am01.reporting as rpt

sns.set_theme(style='whitegrid', context='notebook')

base_config = load_config('configs/ae_mlp.yaml')
base_config['windowing']['window_length'] = PRIMARY_WINDOW_LENGTH
base_config['windowing']['stride'] = PRIMARY_STRIDE
with (CONFIG_DIR / 'official_config.json').open('w', encoding='utf-8') as f:
    json.dump(base_config, f, indent=2)

print('Selection metric:', SELECTION_METRIC)
print('Sensitivity windows:', SENSITIVITY_WINDOWS)

Selection metric: val_pr_auc
Sensitivity windows: [32, 64]


## 3. Dataset inspection e figure iniziali

Genera le figure report-ready sul dataset, inclusa la sensitivity del numero di finestre per `w=32` e `w=64`.

In [5]:
if RUN_REPORT_FIGURES:
    dataset_artifacts = rpt.dataset_report_artifacts(
        base_config,
        DATA_DIR,
        tables_dir=TABLES_DIR,
        figures_dir=FIGURES_DIR,
        window_lengths=SENSITIVITY_WINDOWS,
        primary_window_length=PRIMARY_WINDOW_LENGTH,
    )
    display(dataset_artifacts['dataset_composition'])
    display(dataset_artifacts['window_count_sensitivity'])
else:
    dataset_artifacts = {}

,source,class_label,rows,segments
0,normal,normal,233792,2340
1,slow,slow/anomalous,41538,303


,window_length,train_windows,val_windows,test_windows,test_anomalous_windows,test_anomaly_prevalence,contributing_test_runs
0,32,7892,2595,2619,440,0.168003,528
1,64,4725,1537,1564,320,0.204604,527


## 4. Preprocessing ufficiale

Data loading, split per run, scaling fit solo sui normali di training e windowing ufficiale.

In [6]:
if RUN_PREPROCESSING:
    sh(f'python scripts/audit_data.py --config configs/ae_mlp.yaml --data "{DATA_DIR}" --output "{OFFICIAL_ROOT / "data_audit"}"')
    sh(f'python scripts/prepare_data.py --config configs/ae_mlp.yaml --data "{DATA_DIR}" --output "{OFFICIAL_ROOT / "preprocessed"}"')

preprocessing_summary = rpt.preprocessing_summary_table(
    OFFICIAL_ROOT / 'preprocessed',
    TABLES_DIR / 'preprocessing_summary.csv',
)
display(preprocessing_summary.style.format(precision=3))


$ python scripts/audit_data.py --config configs/ae_mlp.yaml --data "/content/drive/MyDrive/AM01/data/KukaVelocityDataset" --output "/content/drive/MyDrive/AM01/results/official/data_audit"

$ python scripts/prepare_data.py --config configs/ae_mlp.yaml --data "/content/drive/MyDrive/AM01/data/KukaVelocityDataset" --output "/content/drive/MyDrive/AM01/results/official/preprocessed"


,split,rows,runs,windows,anomalous_windows,anomaly_percent
0,train,165495,1586,4725,924,0.196
1,val,54623,529,1537,315,0.205
2,test,55212,528,1564,320,0.205


## 5. Main protocol

Modelli principali: PCA, Isolation Forest, AE MLP, AAE MLP, AE Conv1D.

In [7]:
main_configs = 'configs/pca.yaml configs/isolation_forest.yaml configs/ae_mlp.yaml configs/aae_mlp.yaml configs/ae_conv1d.yaml'
main_output = RUNS_DIR / 'main'
if RUN_MAIN_EXPERIMENTS:
    sh(
        f'python scripts/run_experiments.py '
        f'--configs {main_configs} '
        f'--data "{DATA_DIR}" '
        f'--output "{main_output}" '
        f'--seeds 42 '
        f'--skip-existing '
        f'--summary-name main_results_raw.csv'
    )

main_results = rpt.save_main_results(main_output, TABLES_DIR)
display(main_results.style.format(precision=4))

if RUN_REPORT_FIGURES:
    rpt.plot_main_result_figures(main_output, FIGURES_DIR)


$ python scripts/run_experiments.py --configs configs/pca.yaml configs/isolation_forest.yaml configs/ae_mlp.yaml configs/aae_mlp.yaml configs/ae_conv1d.yaml --data "/content/drive/MyDrive/AM01/data/KukaVelocityDataset" --output "/content/drive/MyDrive/AM01/results/official/runs/main" --seeds 42 --skip-existing --summary-name main_results_raw.csv


,model,run_name,test_f1,test_pr_auc,test_roc_auc,test_precision,test_recall,test_event_recall,test_false_alarms_per_run,val_pr_auc,val_f1,test_windows,test_anomaly_prevalence
3,Isolation Forest,isolation_forest_seed42,0.7156,0.7196,0.9370,0.6163,0.8531,0.9000,0.1499,0.7911,0.7216,1564,0.2046
1,AE Conv1D,ae_conv1d_seed42,0.6106,0.4572,0.8281,0.5396,0.7031,0.6833,0.1784,0.4016,0.4985,1564,0.2046
2,AE MLP,ae_mlp_seed42,0.6109,0.4570,0.8481,0.4683,0.8781,0.9500,0.3017,0.4195,0.5593,1564,0.2046
0,AAE MLP,aae_mlp_seed42,0.5797,0.4437,0.8266,0.4407,0.8469,0.8833,0.3397,0.3910,0.5152,1564,0.2046
4,PCA,pca_seed42,0.3934,0.2873,0.6883,0.3034,0.5594,0.9000,0.4459,0.2974,0.4456,1564,0.2046


## 6. AE vs AAE direct comparison

Questa e' la sezione direttamente collegata alla research question.

In [8]:
ae_vs_aae = rpt.save_ae_vs_aae_comparison(main_results, TABLES_DIR, FIGURES_DIR)
display(ae_vs_aae.style.format(precision=4))

,metric,AE MLP,AAE MLP,AAE_minus_AE
0,test_f1,0.6109,0.5797,-0.0312
1,test_pr_auc,0.4570,0.4437,-0.0133
2,test_roc_auc,0.8481,0.8266,-0.0216
3,test_false_alarms_per_run,0.3017,0.3397,0.0380


## 7. Ablation essenziali

Ablation incluse nel notebook ufficiale:

- window length: `32`, `64`;
- AAE: `latent_dim in {16, 32}`, `lambda_adv in {0.001, 0.01, 0.05, 0.1}`;
- preprocessing/loss: StandardScaler/RobustScaler e MSE/Huber.

In [9]:
window_output = RUNS_DIR / 'window_sensitivity'
aae_ablation_output = RUNS_DIR / 'aae_ablation'
preprocessing_output = RUNS_DIR / 'preprocessing_loss'

if RUN_CORE_ABLATIONS:
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/ae_mlp.yaml configs/aae_mlp.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{window_output}" '
        f'--seeds 42 '
        f'--window-lengths 32 64 '
        f'--skip-existing '
        f'--summary-name window_length_sensitivity_raw.csv'
    )
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/aae_mlp.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{aae_ablation_output}" '
        f'--seeds 42 '
        f'--latent-dims 16 32 '
        f'--lambda-advs 0.001 0.01 0.05 0.1 '
        f'--skip-existing '
        f'--summary-name aae_ablation_raw.csv'
    )
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/ae_mlp.yaml configs/aae_mlp.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{preprocessing_output}" '
        f'--seeds 42 '
        f'--scalers standard robust '
        f'--losses mse huber '
        f'--skip-existing '
        f'--summary-name preprocessing_loss_raw.csv'
    )

ablation_tables = rpt.save_ablation_tables_and_figures(
    window_runs_root=window_output,
    aae_runs_root=aae_ablation_output,
    preprocessing_runs_root=preprocessing_output,
    tables_dir=TABLES_DIR,
    figures_dir=FIGURES_DIR,
)
for name, table in ablation_tables.items():
    display(Markdown(f'### {name}'))
    display(table.head(30).style.format(precision=4))


$ python scripts/run_experiments.py --configs configs/ae_mlp.yaml configs/aae_mlp.yaml --data "/content/drive/MyDrive/AM01/data/KukaVelocityDataset" --output "/content/drive/MyDrive/AM01/results/official/runs/window_sensitivity" --seeds 42 --window-lengths 32 64 --skip-existing --summary-name window_length_sensitivity_raw.csv

$ python scripts/run_experiments.py --configs configs/aae_mlp.yaml --data "/content/drive/MyDrive/AM01/data/KukaVelocityDataset" --output "/content/drive/MyDrive/AM01/results/official/runs/aae_ablation" --seeds 42 --latent-dims 16 32 --lambda-advs 0.001 0.01 0.05 0.1 --skip-existing --summary-name aae_ablation_raw.csv

$ python scripts/run_experiments.py --configs configs/ae_mlp.yaml configs/aae_mlp.yaml --data "/content/drive/MyDrive/AM01/data/KukaVelocityDataset" --output "/content/drive/MyDrive/AM01/results/official/runs/preprocessing_loss" --seeds 42 --scalers standard robust --losses mse huber --skip-existing --summary-name preprocessing_loss_raw.csv


### window_length

,window_length,model,test_windows,test_anomaly_prevalence,test_pr_auc,test_f1,test_roc_auc,val_pr_auc
0,32,AAE MLP,2619,0.1680,0.3502,0.5058,0.7881,0.2925
2,32,AE MLP,2619,0.1680,0.3787,0.5148,0.8249,0.3174
1,64,AAE MLP,1564,0.2046,0.4437,0.5797,0.8266,0.3910
3,64,AE MLP,1564,0.2046,0.4570,0.6109,0.8481,0.4195


### aae_ablation

,run_name,run_dir,model_key,model,seed,window_length,stride,latent_dim,lambda_adv,scaler,loss,threshold,val_threshold,val_precision,val_recall,val_f1,val_balanced_accuracy,val_tn,val_fp,val_fn,val_tp,val_false_positive_rate,val_false_negative_rate,val_roc_auc,val_pr_auc,val_event_recall,val_event_precision,val_true_events,val_predicted_events,val_false_predicted_events,val_false_alarms_per_run,val_mean_false_alarm_duration_windows,val_mean_detection_delay,test_threshold,test_precision,test_recall,test_f1,test_balanced_accuracy,test_tn,test_fp,test_fn,test_tp,test_false_positive_rate,test_false_negative_rate,test_roc_auc,test_pr_auc,test_event_recall,test_event_precision,test_true_events,test_predicted_events,test_false_predicted_events,test_false_alarms_per_run,test_mean_false_alarm_duration_windows,test_mean_detection_delay,test_windows,test_anomalous_windows,test_anomaly_prevalence
0,aae_mlp_seed42_z16_lam0.001,/content/drive/MyDrive/AM01/results/official/runs/aae_ablation/aae_mlp_seed42_z16_lam0.001,aae_mlp,AAE MLP,42,64,16,16,0.0010,standard,mse,0.3057,0.3057,0.3577,0.8381,0.5014,0.7251,748.0000,474.0000,51.0000,264.0000,0.3879,0.1619,0.7754,0.3986,0.9344,0.2126,61.0000,301.0000,237.0000,0.4480,2.0000,2.5263,0.3057,0.4086,0.8875,0.5596,0.7786,833.0000,411.0000,36.0000,284.0000,0.3304,0.1125,0.8326,0.4551,0.9333,0.2097,60.0000,267.0000,211.0000,0.4004,1.9479,2.0000,1564,320,0.2046
1,aae_mlp_seed42_z16_lam0.01,/content/drive/MyDrive/AM01/results/official/runs/aae_ablation/aae_mlp_seed42_z16_lam0.01,aae_mlp,AAE MLP,42,64,16,16,0.0100,standard,mse,0.3092,0.3092,0.3736,0.8349,0.5162,0.7370,781.0000,441.0000,52.0000,263.0000,0.3609,0.1651,0.7807,0.4001,0.9180,0.2318,61.0000,289.0000,222.0000,0.4197,1.9865,2.5714,0.3092,0.4137,0.8688,0.5605,0.7760,850.0000,394.0000,42.0000,278.0000,0.3167,0.1313,0.8367,0.4537,0.9167,0.2129,60.0000,263.0000,207.0000,0.3928,1.9034,2.9091,1564,320,0.2046
2,aae_mlp_seed42_z16_lam0.05,/content/drive/MyDrive/AM01/results/official/runs/aae_ablation/aae_mlp_seed42_z16_lam0.05,aae_mlp,AAE MLP,42,64,16,16,0.0500,standard,mse,0.3406,0.3406,0.3820,0.7397,0.5038,0.7156,845.0000,377.0000,82.0000,233.0000,0.3085,0.2603,0.7672,0.3826,0.8525,0.2421,61.0000,252.0000,191.0000,0.3611,1.9738,5.2308,0.3406,0.4395,0.8281,0.5742,0.7782,906.0000,338.0000,55.0000,265.0000,0.2717,0.1719,0.8208,0.4360,0.8500,0.2321,60.0000,224.0000,172.0000,0.3264,1.9651,3.4510,1564,320,0.2046
3,aae_mlp_seed42_z16_lam0.1,/content/drive/MyDrive/AM01/results/official/runs/aae_ablation/aae_mlp_seed42_z16_lam0.1,aae_mlp,AAE MLP,42,64,16,16,0.1000,standard,mse,0.3306,0.3306,0.3844,0.7810,0.5152,0.7293,828.0000,394.0000,69.0000,246.0000,0.3224,0.2190,0.7747,0.3910,0.8852,0.2423,61.0000,260.0000,197.0000,0.3724,2.0000,2.9630,0.3306,0.4407,0.8469,0.5797,0.7852,900.0000,344.0000,49.0000,271.0000,0.2765,0.1531,0.8266,0.4437,0.8833,0.2350,60.0000,234.0000,179.0000,0.3397,1.9218,3.3208,1564,320,0.2046
4,aae_mlp_seed42_z32_lam0.001,/content/drive/MyDrive/AM01/results/official/runs/aae_ablation/aae_mlp_seed42_z32_lam0.001,aae_mlp,AAE MLP,42,64,16,32,0.0010,standard,mse,0.3214,0.3214,0.3794,0.7841,0.5114,0.7268,818.0000,404.0000,68.0000,247.0000,0.3306,0.2159,0.7725,0.3947,0.8525,0.2299,61.0000,261.0000,201.0000,0.3800,2.0100,2.4615,0.3214,0.4347,0.8531,0.5759,0.7839,889.0000,355.0000,47.0000,273.0000,0.2854,0.1469,0.8250,0.4458,0.9167,0.2254,60.0000,244.0000,189.0000,0.3586,1.8783,3.4909,1564,320,0.2046
5,aae_mlp_seed42_z32_lam0.01,/content/drive/MyDrive/AM01/results/official/runs/aae_ablation/aae_mlp_seed42_z32_lam0.01,aae_mlp,AAE MLP,42,64,16,32,0.0100,standard,mse,0.3201,0.3201,0.3954,0.8222,0.5340,0.7491,826.0000,396.0000,56.0000,259.0000,0.3241,0.1778,0.7875,0.4157,0.9508,0.2472,61.0000,271.0000,204.0000,0.3856,1.9412,3.8621,0.3201,0.4383,0.8656,0.5819,0.7901,889.0000,355.0000,43.0000,277.0000,0.2854,0.1344,0.8380,0.4638,0.9000,0.2282,60.0000,241.0000,186.0000,0.3529,1.9086,2.3704,1564,320,0.2046
6,aae_mlp_seed42_z32_lam0.05,/content/drive/MyDrive/A

### preprocessing_loss

,run_name,run_dir,model_key,model,seed,window_length,stride,latent_dim,lambda_adv,scaler,loss,threshold,val_threshold,val_precision,val_recall,val_f1,val_balanced_accuracy,val_tn,val_fp,val_fn,val_tp,val_false_positive_rate,val_false_negative_rate,val_roc_auc,val_pr_auc,val_event_recall,val_event_precision,val_true_events,val_predicted_events,val_false_predicted_events,val_false_alarms_per_run,val_mean_false_alarm_duration_windows,val_mean_detection_delay,test_threshold,test_precision,test_recall,test_f1,test_balanced_accuracy,test_tn,test_fp,test_fn,test_tp,test_false_positive_rate,test_false_negative_rate,test_roc_auc,test_pr_auc,test_event_recall,test_event_precision,test_true_events,test_predicted_events,test_false_predicted_events,test_false_alarms_per_run,test_mean_false_alarm_duration_windows,test_mean_detection_delay,test_windows,test_anomalous_windows,test_anomaly_prevalence
0,aae_mlp_seed42_scalerrobust_losshuber,/content/drive/MyDrive/AM01/results/official/runs/preprocessing_loss/aae_mlp_seed42_scalerrobust_losshuber,aae_mlp,AAE MLP,42,64,16,16,0.1000,robust,huber,2.2728,2.2728,0.4471,0.7238,0.5527,0.7465,940.0000,282.0000,87.0000,228.0000,0.2308,0.2762,0.7931,0.3969,1.0000,0.3750,61.0000,216.0000,135.0000,0.2552,2.0889,5.5082,2.2728,0.4066,0.6188,0.4907,0.6932,955.0000,289.0000,122.0000,198.0000,0.2323,0.3812,0.7571,0.3370,0.9500,0.3612,60.0000,227.0000,145.0000,0.2751,1.9931,11.5088,1564,320,0.2046
1,aae_mlp_seed42_scalerrobust_lossmse,/content/drive/MyDrive/AM01/results/official/runs/preprocessing_loss/aae_mlp_seed42_scalerrobust_lossmse,aae_mlp,AAE MLP,42,64,16,16,0.1000,robust,mse,58.8508,58.8508,0.4144,0.8063,0.5474,0.7563,863.0000,359.0000,61.0000,254.0000,0.2938,0.1937,0.7765,0.3510,1.0000,0.3253,61.0000,249.0000,168.0000,0.3176,2.1369,1.3115,58.8508,0.3873,0.7406,0.5086,0.7196,869.0000,375.0000,83.0000,237.0000,0.3014,0.2594,0.7729,0.3537,1.0000,0.3270,60.0000,263.0000,177.0000,0.3359,2.1186,1.6000,1564,320,0.2046
2,aae_mlp_seed42_scalerstandard_losshuber,/content/drive/MyDrive/AM01/results/official/runs/preprocessing_loss/aae_mlp_seed42_scalerstandard_losshuber,aae_mlp,AAE MLP,42,64,16,16,0.1000,standard,huber,0.1501,0.1501,0.6027,0.7079,0.6511,0.7938,1075.0000,147.0000,92.0000,223.0000,0.1203,0.2921,0.8786,0.6921,0.8197,0.4444,61.0000,135.0000,75.0000,0.1418,1.9600,4.4800,0.1501,0.6126,0.7906,0.6903,0.8310,1084.0000,160.0000,67.0000,253.0000,0.1286,0.2094,0.9035,0.7080,0.7833,0.4370,60.0000,119.0000,67.0000,0.1271,2.3881,3.0638,1564,320,0.2046
3,aae_mlp_seed42_scalerstandard_lossmse,/content/drive/MyDrive/AM01/results/official/runs/preprocessing_loss/aae_mlp_seed42_scalerstandard_lossmse,aae_mlp,AAE MLP,42,64,16,16,0.1000,standard,mse,0.3306,0.3306,0.3844,0.7810,0.5152,0.7293,828.0000,394.0000,69.0000,246.0000,0.3224,0.2190,0.7747,0.3910,0.8852,0.2423,61.0000,260.0000,197.0000,0.3724,2.0000,2.9630,0.3306,0.4407,0.8469,0.5797,0.7852,900.0000,344.0000,49.0000,271.0000,0.2765,0.1531,0.8266,0.4437,0.8833,0.2350,60.0000,234.0000,179.0000,0.3397,1.9218,3.3208,1564,320,0.2046
4,ae_mlp_seed42_scalerrobust_losshuber,/content/drive/MyDrive/AM01/results/official/runs/preprocessing_loss/ae_mlp_seed42_scalerrobust_losshuber,ae_mlp,AE MLP,42,64,16,16,nan,robust,huber,1.7896,1.7896,0.5874,0.8857,0.7063,0.8627,1026.0000,196.0000,36.0000,279.0000,0.1604,0.1143,0.8944,0.5340,1.0000,0.4969,61.0000,161.0000,81.0000,0.1531,2.4198,1.8361,1.7896,0.5700,0.8906,0.6951,0.8589,1029.0000,215.0000,35.0000,285.0000,0.1728,0.1094,0.8682,0.4600,1.0000,0.4647,60.0000,170.0000,91.0000,0.1727,2.3626,1.8667,1564,320,0.2046
5,ae_mlp_seed42_scalerrobust_lossmse,/content/drive/MyDrive/AM01/results/official/runs/preprocessing_loss/ae_mlp_seed42_scalerrobust_lossmse,ae_mlp,AE MLP,42,64,16,16,nan,robust,mse,34.6648,34.6648,0.4690,0.9111,0.6192,0.8226,897.0000,325.0000,28.0000,287.0000,0.2660,0.0889,0.8134,0.3745,1.0000,0.3453,61.0000,223.0000,146.0000,0.2760,2.2260,0.5246,34.6648,0.4729,0.8719,0.6132,0.8109,933.0000,311.0000,41.0000,279.0000,0.25

## 8. AAE-specific scoring e latent diagnostics

Il run AAE viene selezionato usando `val_pr_auc`, non il test. Qui verifichiamo se latent space e discriminator contengono segnale utile oltre alla reconstruction error.

In [10]:
def collect_existing_metrics(roots):
    frames = []
    for root in roots:
        root = Path(root)
        if root.exists():
            try:
                frames.append(rpt.collect_run_metrics(root))
            except FileNotFoundError:
                pass
    if not frames:
        raise FileNotFoundError('Nessun run disponibile per la selezione AE/AAE.')
    return pd.concat(frames, ignore_index=True)

candidate_runs = collect_existing_metrics([main_output, aae_ablation_output, preprocessing_output])
candidate_runs.to_csv(TABLES_DIR / 'candidate_ae_aae_runs.csv', index=False)

ae_candidates = candidate_runs[candidate_runs['model_key'] == 'ae_mlp'].dropna(subset=[SELECTION_METRIC])
aae_candidates = candidate_runs[candidate_runs['model_key'] == 'aae_mlp'].dropna(subset=[SELECTION_METRIC])
BEST_AE_RUN = Path(ae_candidates.sort_values(SELECTION_METRIC, ascending=False).iloc[0]['run_dir'])
BEST_AAE_RUN = Path(aae_candidates.sort_values(SELECTION_METRIC, ascending=False).iloc[0]['run_dir'])

display(Markdown(f'**Selected AE:** `{BEST_AE_RUN}`  \n**Selected AAE:** `{BEST_AAE_RUN}`'))

if RUN_AAE_DIAGNOSTICS:
    diag = rpt.aae_diagnostics_artifacts(
        ae_run_dir=BEST_AE_RUN,
        aae_run_dir=BEST_AAE_RUN,
        tables_dir=TABLES_DIR,
        figures_dir=FIGURES_DIR,
        extended_dir=EXTENDED_DIR,
        selection_metric=SELECTION_METRIC,
    )
    display(diag['score_table'].style.format(precision=4))
    if not diag['per_feature'].empty:
        display(diag['per_feature'].sort_values('delta').head(20).style.format(precision=5))
    if not diag['per_action'].empty:
        display(diag['per_action'].head(30).style.format(precision=4))
else:
    diag = {'score_table': pd.read_csv(TABLES_DIR / 'aae_specific_scores.csv')}

**Selected AE:** `/content/drive/MyDrive/AM01/results/official/runs/preprocessing_loss/ae_mlp_seed42_scalerstandard_losshuber`  
**Selected AAE:** `/content/drive/MyDrive/AM01/results/official/runs/preprocessing_loss/aae_mlp_seed42_scalerstandard_losshuber`

,aae_score,threshold,val_pr_auc,val_roc_auc,test_pr_auc,test_roc_auc,test_f1,test_balanced_accuracy,test_false_alarms_per_run
0,score_rec,0.3343,0.3902,0.7700,0.4483,0.8230,0.5404,0.7578,0.3852
9,score_combined_rec_latent_a0p75,0.2969,0.3361,0.7075,0.4108,0.7784,0.5565,0.7451,0.2713
8,score_combined_rec_disc_a0p75,0.2382,0.3213,0.6999,0.3817,0.7623,0.4519,0.6784,0.5769
7,score_combined_rec_latent_a0p5,0.1639,0.2493,0.5617,0.3073,0.6319,0.3493,0.5251,0.8558
6,score_combined_rec_disc_a0p5,0.1618,0.2065,0.5340,0.2289,0.5786,0.3582,0.5394,0.8672
5,score_combined_rec_latent_a0p25,0.0477,0.1984,0.4299,0.2219,0.4718,0.3397,0.5000,0.8861
1,score_latent_norm,3.0231,0.1896,0.3970,0.1830,0.3914,0.3390,0.4988,0.8861
2,score_latent_mahalanobis,4.0766,0.1773,0.3664,0.1790,0.3809,0.3381,0.4973,0.8861
4,score_combined_rec_disc_a0p25,0.1578,0.1563,0.3875,0.1545,0.3796,0.3477,0.5196,0.8767
3,score_disc,0.4414,0.1428,0.3213,0.1369,0.2880,0.3342,0.4947,0.8824


,feature,normal_error_ae,anomaly_error_ae,separation_ae,normal_error_aae,anomaly_error_aae,separation_aae,delta
46,sensor_id4_GyroZ,0.11424,0.18718,0.07295,0.32274,0.20698,-0.11576,-0.18871
45,sensor_id4_GyroY,0.11860,0.20851,0.08991,0.31135,0.22579,-0.08556,-0.17547
82,sensor_id7_q3,0.04515,0.83384,0.78870,0.07360,0.70947,0.63587,-0.15283
65,sensor_id6_AccZ,0.70028,0.32881,-0.37148,0.85530,0.34651,-0.50879,-0.13731
32,sensor_id3_AccZ,0.72997,0.55777,-0.17221,0.85754,0.56857,-0.28898,-0.11677
16,sensor_id1_q3,0.04774,0.28237,0.23463,0.08049,0.19888,0.11839,-0.11623
81,sensor_id7_q2,0.04956,0.64849,0.59893,0.08014,0.56474,0.48460,-0.11433
64,sensor_id6_AccY,0.48628,0.38729,-0.09899,0.60774,0.42283,-0.18491,-0.08591
5,machine_nameKuka Robot_power_factor,0.50052,1.29537,0.79484,0.64454,1.36049,0.71595,-0.07890
36,sensor_id3_q1,0.04064,0.17593,0.13529,0.07418,0.13181,0.05763,-0.07766


,model,action,n_windows,anomaly_prevalence,f1,precision,recall,tp,fp,tn,fn
0,AE,0.0000,18,0.0000,0.0000,0.0000,0.0000,0,0,18,0
1,AE,1.0000,50,0.2800,0.8000,0.6667,1.0000,14,7,29,0
2,AE,2.0000,60,0.2333,0.6512,0.4828,1.0000,14,15,31,0
3,AE,3.0000,62,0.2903,0.7083,0.5667,0.9444,17,13,31,1
4,AE,4.0000,85,0.4824,0.7711,0.7619,0.7805,32,10,34,9
5,AE,5.0000,59,0.0000,0.0000,0.0000,0.0000,0,29,30,0
6,AE,6.0000,49,0.0816,0.4211,0.2667,1.0000,4,11,34,0
7,AE,7.0000,54,0.1667,0.1333,0.1667,0.1111,1,5,40,8
8,AE,8.0000,106,0.1981,0.5846,0.4318,0.9048,19,25,60,2
9,AE,9.0000,40,0.0000,0.0000,0.0000,0.0000,0,11,29,0


## 9. Summary finale e manifest artefatti

In [11]:
aae_specific = pd.read_csv(TABLES_DIR / 'aae_specific_scores.csv') if (TABLES_DIR / 'aae_specific_scores.csv').exists() else None
window_sensitivity = pd.read_csv(TABLES_DIR / 'window_length_sensitivity.csv') if (TABLES_DIR / 'window_length_sensitivity.csv').exists() else None
rpt.write_official_summary(
    output_path=OFFICIAL_ROOT / 'summary.md',
    main_results=main_results,
    ae_vs_aae=ae_vs_aae,
    aae_specific=aae_specific,
    window_sensitivity=window_sensitivity,
)
display(Markdown((OFFICIAL_ROOT / 'summary.md').read_text(encoding='utf-8')))

artifacts = sorted([p for p in OFFICIAL_ROOT.rglob('*') if p.is_file()])
manifest = pd.DataFrame({'artifact': [str(p.relative_to(OFFICIAL_ROOT)) for p in artifacts]})
manifest.to_csv(TABLES_DIR / 'artifact_manifest.csv', index=False)
display(manifest)

# AM01 Official Summary

- Best main model by test PR-AUC: **Isolation Forest** (`PR-AUC=0.7196`).
- AE vs AAE delta under the main protocol: `PR-AUC=-0.0133`, `F1=-0.0312`.
- Main protocol uses `window_length=64`, `stride=16`, StandardScaler, MSE and seed 42.
- Window sensitivity is limited to `w=32` and `w=64`; `w=128` is intentionally excluded because it leaves too few test windows.
- Best AAE-specific score by validation PR-AUC: `score_rec` (`val PR-AUC=0.3902`, `test PR-AUC=0.4483`).

## Window Sensitivity

| window_length | model | test_windows | test_pr_auc |
| --- | --- | --- | --- |
| 32 | AAE MLP | 2619 | 0.3502 |
| 32 | AE MLP | 2619 | 0.3787 |
| 64 | AAE MLP | 1564 | 0.4437 |
| 64 | AE MLP | 1564 | 0.4570 |

## Final Takeaways

1. The dataset must be evaluated as temporal windows; window length materially changes the evaluation population.
2. Under the official `w=64` protocol, AAE does not robustly improve the AE baseline.
3. AAE-specific latent/discriminator scores are useful diagnostics, but they are treated as supporting analysis rather than the primary comparison.
4. The final conclusion is critical: adversarial regularization is not automatically beneficial for this Kuka anomaly-detection setting.


,artifact
0,config/environment.txt
1,config/official_config.json
2,data_audit/dataset_summary.csv
3,data_audit/feature_summary.csv
4,extended_scores/aae_extended_scores_test.csv
...,...
399,tables/preprocessing_loss_ablation.csv
400,tables/preprocessing_summary.csv
401,tables/segment_lengths.csv
402,tables/window_count_sensitivity.csv
